In [ ]:
#############################
### Overview of the code: ###
#############################

# Image classification using Hugging Face Transformers
# First we will build MNIST image classification using PyTorch. 
# This will explain the basics of using Hugging Face Transformers for image classification.
# Later we will expand this to more complex datasets and models.
# I am planning to use the CFIAR-10 dataset for image classification. 
# It has 10 classes of images and is a common benchmark for image classification tasks.
# We will build basic logistic regression model using PyTorch for CFIAR-10 dataset.
# We will show challenges of building image classification models using Logistic Regression.
# We will display learning curves for Logistic Regression model. 
# And accuracy on test set will be calculated.
# After that we will build a simple Deeplearning model for image classification using PyTorch.
# we wills show differences between Logistic Regression and Deep Learning models for image classification.
# We will also show learning curves and test accuracy for Deep Learning model.
# And then we will build CNN model for image classification using PyTorch.
# We will show how CNNs are better than Logistic Regression and Deep Learning models for image classification
# We will also show learning curves and test accuracy for CNN model.
# Show the differences in performance between Logistic Regression, Deep Learning and 
# CNN models for image classification.
# And the finally we will build a ResNet model for image classification using PyTorch.
# We will show how ResNet is better than Logistic Regression, Deep Learning, 
# CNN and resnet models for image classification.
# We will also show learning curves and test accuracy for ResNet model.

# Finally we will data augmentation and regularization techniques for image classification using PyTorch.
# We will show how data augmentation and regularization can improve performance of image classification models.
# We will perform data augmentation using techniques like random cropping, flipping, and rotation.
# We will also show how regularization techniques like dropout and weight decay 
# can improve performance of image classification models.
# After applying data augmentation and regularization techniques we will 
# re-evaluate the performance of all the models and show the improvements in learning curves and test accuracy.

# Let us build code in modular way so that we can easily reuse the code for different models and datasets. 
# Also we re-evaluate the performance of all the models after applying data augmentation and 
# regularization techniques.

In [ ]:
# Let us start with MNIST image classification using Logistic Regression model.
# import necessary libraries
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
from torch.utils.tensorboard import SummaryWriter
from torchvision.datasets import MNIST
from matplotlib import pyplot as plt
import numpy as np


In [ ]:
# define the device to be used for training
def get_device():
    return torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
# let us define the parameters for training
def model_parameters():
    params = {
        "batch_size": 64,
        "learning_rate": 0.01,
        "num_epochs": 10,
        "device": get_device()
    }
    return params

In [ ]:
# let define MNIST specific parameters
def mnist_parameters():
    params = model_parameters()
    params["input_size"] = 28 * 28
    params["num_classes"] = 10
    params["dataset"] = "MNIST"
    return params

In [ ]:
# let def CFIAR-10 specific parameters
def cifar10_parameters():
    params = model_parameters()
    params["input_size"] = 32 * 32 * 3
    params["num_classes"] = 10
    params["dataset"] = "CFIAR-10"
    return params

In [ ]:
# def resnet specific parameters
def resnet_parameters():
    params = model_parameters()
    params["input_size"] = 32 * 32 * 3
    params["num_classes"] = 10
    params["dataset"] = "CFIAR-10"
    params["model"] = "ResNet"
    params["num_epochs"] = 50
    return params

In [ ]:
# Load the MNIST dataset both training and test sets
def load_mnist_data(params):
    train_dataset = MNIST(root='./data', train=True, download=True, transform=transforms.ToTensor())
    test_dataset = MNIST(root='./data', train=False, download=True, transform=transforms.ToTensor())
    train_loader = DataLoader(train_dataset, batch_size=params["batch_size"], shuffle=True)
    test_loader = DataLoader(test_dataset, batch_size=params["batch_size"], shuffle=False)
    return train_loader, test_loader

In [ ]:
# let define data augmentation and regularization techniques for image classification using PyTorch
# we will perform data augmentation using techniques like random cropping, flipping, and rotation.
# we will also show how regularization techniques like dropout and weight decay can help prevent overfitting.

# let us define the data augmentation techniques for CFIAR-10 dataset for logistic regression model
def data_augmentation_cifar10():
    transform = transforms.Compose([
        transforms.RandomHorizontalFlip(),
        transforms.RandomCrop(32, padding=4),
        transforms.RandomRotation(15),
        transforms.ToTensor(),
        transforms.RandomErasing(p=0.5)
    ])
    return transform


In [ ]:
# load the CFIAR-10 dataset both training and test sets
def load_cifar10_data(params, data_transformation=None):
    train_dataset = datasets.CIFAR10(root='./data', train=True, download=True, 
                                     transform=transforms.ToTensor() if data_transformation is None else data_transformation)
    test_dataset = datasets.CIFAR10(root='./data', train=False, download=True, 
                                    transform=transforms.ToTensor() if data_transformation is None else data_transformation)
    train_loader = DataLoader(train_dataset, batch_size=params["batch_size"], shuffle=True)
    test_loader = DataLoader(test_dataset, batch_size=params["batch_size"], shuffle=False)
    return train_loader, test_loader


In [ ]:
# split the train dataset into train and validation sets
def split_train_val(train_loader, params, val_split=0.2):
    train_size = int(len(train_loader.dataset) * (1 - val_split))
    val_size = len(train_loader.dataset) - train_size
    train_dataset, val_dataset = torch.utils.data.random_split(train_loader.dataset, [train_size, val_size])
    train_loader = DataLoader(train_dataset, batch_size=params["batch_size"], shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=params["batch_size"], shuffle=False)
    return train_loader, val_loader

In [ ]:
# Show the sample images from the dataset and size of the dataset
def show_sample_images(train_loader):
    for images, labels in train_loader:
        fig, axes = plt.subplots(1, 5, figsize=(10, 2))
        for i in range(5):
            axes[i].imshow(images[i].squeeze(), cmap='gray')
            axes[i].set_title(f'Label: {labels[i].item()}')
            axes[i].axis('off')
        plt.show()
        break
    print(f"Number of training examples: {len(train_loader.dataset)}")
    return

In [ ]:
# calculate the accuracy of the model on valudation and test sets. Also calculate loss on test set
def calculate_metrics(model, val_loader, test_loader, criterion, params):
    model.to(params["device"])
    with torch.no_grad():
        model.eval()
        val_correct = 0
        val_total = 0
        val_loss = 0
        for images, labels in val_loader:
            images = images.to(params["device"])
            labels = labels.to(params["device"])
            outputs = model(images)
            loss = criterion(outputs, labels)
            val_loss += loss.item()
            _, predicted = torch.max(outputs.data, 1)
            val_total += labels.size(0)
            val_correct += (predicted == labels).sum().item()
        val_accuracy = 100 * val_correct / val_total

        val_loss = val_loss / len(val_loader)

        test_correct = 0
        test_total = 0
        test_loss = 0
        for images, labels in test_loader:
            images = images.to(params["device"])
            labels = labels.to(params["device"])
            outputs = model(images)
            loss = criterion(outputs, labels)
            test_loss += loss.item()
            _, predicted = torch.max(outputs.data, 1)
            test_total += labels.size(0)
            test_correct += (predicted == labels).sum().item()
        test_accuracy = 100 * test_correct / test_total
    return val_accuracy, test_accuracy, val_loss, test_loss

In [ ]:
# display the learning curves for training and validation loss and accuracy
def display_learning_curves(train_losses, val_losses, train_accuracies, val_accuracies):
    epochs = range(1, len(train_losses) + 1)
    plt.figure(figsize=(12, 5))
    plt.subplot(1, 2, 1)
    plt.plot(epochs, train_losses, label='Training Loss')
    plt.plot(epochs, val_losses, label='Validation Loss')
    plt.xlabel('Epochs')
    plt.ylabel('Loss')
    plt.title('Loss Curves')
    plt.legend()
    plt.subplot(1, 2, 2)
    plt.plot(epochs, train_accuracies, label='Training Accuracy')
    plt.plot(epochs, val_accuracies, label='Validation Accuracy')
    plt.xlabel('Epochs')
    plt.ylabel('Accuracy (%)')
    plt.title('Accuracy Curves')
    plt.legend()
    plt.show()

In [ ]:
# define the training loop for the model
def train_model(model, train_loader, val_loader, test_loader, params):
    model.to(params["device"])
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.SGD(model.parameters(), lr=params["learning_rate"], momentum=0.9, weight_decay=1e-4)
    train_losses = []
    val_losses = []
    test_losses = []    
    train_accuracies = []
    val_accuracies = []
    test_accuracies = []

    running_loss = 0.0
    for epoch in range(params["num_epochs"]):
        model.train()
        running_loss = 0.0
        for images, labels in train_loader:
            images = images.to(params["device"])
            labels = labels.to(params["device"])
            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            running_loss += loss.item()
            loss.backward()
            optimizer.step()
        val_accuracy, test_accuracy, val_loss, test_loss = calculate_metrics(model, val_loader, test_loader, criterion, params)
        train_losses.append(running_loss / len(train_loader))
        val_losses.append(val_loss)
        test_losses.append(test_loss)
        train_accuracies.append(val_accuracy)
        val_accuracies.append(test_accuracy)
        test_accuracies.append(test_accuracy)
        print(f"Epoch [{epoch + 1}/{params['num_epochs']}], Loss: {loss.item()}, Val Accuracy: {val_accuracy:.2f}, Test Accuracy: {test_accuracy:.2f}")
    return model, train_losses, val_losses, test_losses, train_accuracies, val_accuracies

In [ ]:
# define the function to check predict image and label from the test set
def predict_image(model, test_loader, params):
    model.to(params["device"])
    model.eval()
    with torch.no_grad():
        for images, labels in test_loader:
            images = images.to(params["device"])
            labels = labels.to(params["device"])
            outputs = model(images)
            _, predicted = torch.max(outputs.data, 1)
            # Print first 5 predictions from the batch
            if params["dataset"] == "MNIST":
                for i in range(min(5, len(predicted))):
                    print(f"Predicted label: {predicted[i].item()}, Actual label: {labels[i].item()}")
            elif params["dataset"] == "CFIAR-10":
                classes = ['airplane', 'automobile', 'bird', 'cat', 'deer', 'dog', 'frog', 'horse', 'ship', 'truck']
                for i in range(min(5, len(predicted))):
                    print(f"Predicted label: {classes[predicted[i].item()]}, Actual label: {classes[labels[i].item()]}")
            break
    return
    

In [ ]:
# define the function to check the parameters of the model
def check_model_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

In [ ]:
# Now we have all modules defined for training the model, calculating metrics and displaying learning curves.
# Let us now build the Logistic Regression model for MNIST image classification.

# define the Logistic Regression model for MNIST image classification
class LogisticRegression(nn.Module):
    def __init__(self, input_size, num_classes):
        super(LogisticRegression, self).__init__()
        self.flatten = nn.Flatten()
        self.linear = nn.Linear(input_size, num_classes)

    def forward(self, x):
        out = self.flatten(x)
        out = self.linear(out)
        return out
    

In [ ]:
# create an instance of the Logistic Regression model
def create_logistic_regression_model(params):
    model = LogisticRegression(params["input_size"], params["num_classes"])
    return model

In [ ]:
device = get_device()
print(f"Using device: {device}")

model = create_logistic_regression_model(mnist_parameters())
train_loader, test_loader = load_mnist_data(mnist_parameters())
train_loader, val_loader = split_train_val(train_loader, mnist_parameters())
# show_sample_images(train_loader)
model, train_losses, val_losses, test_losses, train_accuracies, val_accuracies = train_model(model, train_loader, val_loader, test_loader, mnist_parameters())


print(f"Total trainable parameters: {check_model_parameters(model)}")
display_learning_curves(train_losses, val_losses, train_accuracies, val_accuracies)
predict_image(model, test_loader, mnist_parameters())

In [ ]:
# Run the code with CFIAR-10 dataset and Logistic Regression model

def run_logistic_regression_cifar10(data_normalization_techniques=False):
    model = create_logistic_regression_model(cifar10_parameters())
    if data_normalization_techniques:
        train_loader, test_loader = load_cifar10_data(cifar10_parameters(),data_augmentation_cifar10())
    else:
        train_loader, test_loader = load_cifar10_data(cifar10_parameters())
    train_loader, val_loader = split_train_val(train_loader, cifar10_parameters())
    model, train_losses, val_losses, test_losses, train_accuracies, val_accuracies = \
                    train_model(model, train_loader, val_loader, test_loader, cifar10_parameters())

    print(f"Total trainable parameters: {check_model_parameters(model)}")
    display_learning_curves(train_losses, val_losses, train_accuracies, val_accuracies)
    predict_image(model, test_loader, cifar10_parameters())

In [ ]:
# define simple Deep Learning model for CFIAR-10 image classification
class SimpleDeepLearning(nn.Module):
    def __init__(self, input_size, num_classes):
        super(SimpleDeepLearning, self).__init__()
        self.flatten = nn.Flatten()
        self.fc1 = nn.Linear(input_size, 512)
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(512, num_classes)
        self.dropout = nn.Dropout(0.5)

    def forward(self, x):
        x = self.flatten(x)
        x = self.fc1(x)
        x = self.relu(x)
        x = self.dropout(x)
        x = self.fc2(x)
        return x

In [ ]:
# create an instance of the Simple Deep Learning model
def create_simple_deep_learning_model(params):
    model = SimpleDeepLearning(params["input_size"], params["num_classes"])
    return model

In [ ]:
# run the code with CFIAR-10 dataset and Simple Deep Learning model
def run_simple_deep_learning_cifar10(data_normalization_techniques=False):
    model = create_simple_deep_learning_model(cifar10_parameters())
    if data_normalization_techniques:
        train_loader, test_loader = load_cifar10_data(cifar10_parameters(),data_augmentation_cifar10())
    else:
        train_loader, test_loader = load_cifar10_data(cifar10_parameters())
    train_loader, val_loader = split_train_val(train_loader, cifar10_parameters())
    model, train_losses, val_losses, test_losses, train_accuracies, val_accuracies = \
                    train_model(model, train_loader, val_loader, test_loader, cifar10_parameters())

    print(f"Total trainable parameters: {check_model_parameters(model)}")
    display_learning_curves(train_losses, val_losses, train_accuracies, val_accuracies)
    predict_image(model, test_loader, cifar10_parameters())

In [ ]:
# define CNN model for CFIAR-10 image classification
class SimpleCNN(nn.Module):
    def __init__(self, input_size, num_classes):
        super(SimpleCNN, self).__init__()
        self.conv1 = nn.Conv2d(3, 16, kernel_size=3, stride=1, padding=1)
        self.pool = nn.MaxPool2d(kernel_size=2, stride=2, padding=0)
        self.conv2 = nn.Conv2d(16, 32, kernel_size=3, stride=1, padding=1)
        self.conv3 = nn.Conv2d(32, 64, kernel_size=3, stride=1, padding=1)
        self.fc1 = nn.Linear(64 * 4 * 4, 512)
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(512, num_classes)
        self.dropout = nn.Dropout(0.5)

    def forward(self, x):
        x = self.pool(self.relu(self.conv1(x)))
        x = self.pool(self.relu(self.conv2(x)))
        x = self.pool(self.relu(self.conv3(x)))
        x = x.view(-1, 64 * 4 * 4)
        x = self.relu(self.fc1(x))
        x = self.dropout(x)
        x = self.fc2(x)
        return x
    

In [ ]:
# create an instance of the CNN model
def create_cnn_model(params):
    model = SimpleCNN(params["input_size"], params["num_classes"])
    return model

In [ ]:
# run the code with CFIAR-10 dataset and CNN model
def run_cnn_cifar10(data_normalization_techniques=False):
    model = create_cnn_model(cifar10_parameters())
    if data_normalization_techniques:
        train_loader, test_loader = load_cifar10_data(cifar10_parameters(),data_augmentation_cifar10())
    else:
        train_loader, test_loader = load_cifar10_data(cifar10_parameters())
    train_loader, val_loader = split_train_val(train_loader, cifar10_parameters())
    model, train_losses, val_losses, test_losses, train_accuracies, val_accuracies = \
                    train_model(model, train_loader, val_loader, test_loader, cifar10_parameters())

    print(f"Total trainable parameters: {check_model_parameters(model)}")
    display_learning_curves(train_losses, val_losses, train_accuracies, val_accuracies)
    predict_image(model, test_loader, cifar10_parameters())

In [ ]:
# def resnet model for CFIAR-10 image classification
class ResNetBlock(nn.Module):
    def __init__(self, in_channels, out_channels, stride=1):
        super(ResNetBlock, self).__init__()
        self.conv1 = nn.Conv2d(in_channels, out_channels, kernel_size=3, stride=stride, padding=1)
        self.bn1 = nn.BatchNorm2d(out_channels)
        self.relu = nn.ReLU()
        self.conv2 = nn.Conv2d(out_channels, out_channels, kernel_size=3, stride=1, padding=1)
        self.bn2 = nn.BatchNorm2d(out_channels)
        self.downsample = None
        if stride != 1 or in_channels != out_channels:
            self.downsample = nn.Sequential(
                nn.Conv2d(in_channels, out_channels, kernel_size=1, stride=stride),
                nn.BatchNorm2d(out_channels)
            )

    def forward(self, x):
        identity = x
        out = self.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        if self.downsample is not None:
            identity = self.downsample(x)
        out += identity
        out = self.relu(out)
        return out


class SimpleResNet(nn.Module):
    def __init__(self, num_classes):
        super(SimpleResNet, self).__init__()
        self.conv1 = nn.Conv2d(3, 16, kernel_size=3, stride=1, padding=1)
        self.bn1 = nn.BatchNorm2d(16)
        self.relu = nn.ReLU()
        self.block1 = ResNetBlock(16, 32, stride=2)
        self.block2 = ResNetBlock(32, 64, stride=2)
        self.pool = nn.AdaptiveAvgPool2d((1, 1))
        self.fc = nn.Linear(64, num_classes)

    def forward(self, x):
        x = self.relu(self.bn1(self.conv1(x)))
        x = self.block1(x)
        x = self.block2(x)
        x = self.pool(x)
        x = x.view(x.size(0), -1)
        x = self.fc(x)
        return x

In [ ]:
# create an instance of the resnet model for CFIAR-10 image classification
def create_resnet_model(params):
    model = SimpleResNet(params["num_classes"])
    return model

In [ ]:
# run the code with CFIAR-10 dataset and resnet model
def run_resnet_cifar10(data_normalization_techniques=False):
    model = create_resnet_model(resnet_parameters())
    if data_normalization_techniques:
        train_loader, test_loader = load_cifar10_data(resnet_parameters(), data_augmentation_cifar10())
    else:
        train_loader, test_loader = load_cifar10_data(resnet_parameters())
    train_loader, val_loader = split_train_val(train_loader, resnet_parameters())
    model, train_losses, val_losses, test_losses, train_accuracies, val_accuracies = \
                    train_model(model, train_loader, val_loader, test_loader, resnet_parameters())

    print(f"Total trainable parameters: {check_model_parameters(model)}")
    display_learning_curves(train_losses, val_losses, train_accuracies, val_accuracies)
    predict_image(model, test_loader, resnet_parameters())

In [ ]:
# run all models without data augmentation and regularization techniques
print("Running Logistic Regression model on CFIAR-10 dataset without data augmentation and regularization techniques...")
run_logistic_regression_cifar10(data_normalization_techniques=False)
print("Running Deep Learning model on CFIAR-10 dataset without data augmentation and regularization techniques...")
run_simple_deep_learning_cifar10(data_normalization_techniques=False)
print("Running CNN model on CFIAR-10 dataset without data augmentation and regularization techniques...")
run_cnn_cifar10(data_normalization_techniques=False)
print("Running ResNet model on CFIAR-10 dataset without data augmentation and regularization techniques...")
run_resnet_cifar10(data_normalization_techniques=False)

In [ ]:
# run all models with data augmentation and regularization techniques for CFIAR-10 dataset
print("Running Logistic Regression model on CFIAR-10 dataset with data augmentation and regularization techniques...")
data_augmentation_transform = data_augmentation_cifar10()
run_logistic_regression_cifar10(data_normalization_techniques=True)
print("Running Deep Learning model on CFIAR-10 dataset with data augmentation and regularization techniques...")
run_simple_deep_learning_cifar10(data_normalization_techniques=True)
print("Running CNN model on CFIAR-10 dataset with data augmentation and regularization techniques...")
run_cnn_cifar10(data_normalization_techniques=True)
print("Running ResNet model on CFIAR-10 dataset with data augmentation and regularization techniques...")
run_resnet_cifar10(data_normalization_techniques=True)